# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/m-awaischohan/flyrank-internship-ml/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

**Lane lock: I am confirming Lane 4 (CTR / Engagement Opportunity Scoring).** Two reasons from my own numbers: the starter's fixed "CTR < 0.5%" rule flags about 79% of visible pages and is right only about 48% of the time against a 41% base rate (week 2), and the gap between a page's CTR and its position-band norm does seem to persist (signal 2 below). That is room for a ranked list to add something.

**Data for this notebook:** the 30,000-row starter CSV in the repo (one row = one content item, trailing-90-day totals plus two 30-day windows). The rule only reads the **earlier** 30-day window (`*_prev_30d`) and the position average. The later 30 days (`*_last_30d`) are used only afterwards, to check how the picks turned out; they are never an input.

Before I write the rule I check two signals it leans on. **Signal 1 is behind a real FlyRank flag** (`low_ctr_visible_page`: more than 500 impressions, position 20 or better, CTR under 0.5%). Signal 2 is the assumption my own rule rests on.

In [ ]:
import os, sys, subprocess, json
import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/m-awaischohan/flyrank-internship-ml"
REPO_DIR = "flyrank-internship-ml"

if IN_COLAB:
    if os.path.basename(os.getcwd()) != REPO_DIR:
        if not os.path.isdir(REPO_DIR):
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
        os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")
os.makedirs("work/outputs", exist_ok=True)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Pages the rule may look at: enough impressions in the EARLIER 30 days, position 1-20 (0 = "no data")
BANDS = [0, 3, 6, 10, 15, 20]
BAND_LABELS = ["1-3", "4-6", "7-10", "11-15", "16-20"]
el = df[(df["impressions_prev_30d"] >= 100) & (df["avg_position"] > 0) & (df["avg_position"] <= 20)].copy()
el["band"] = pd.cut(el["avg_position"], BANDS, labels=BAND_LABELS)
el["ctr_prev"] = 100 * el["clicks_prev_30d"] / el["impressions_prev_30d"]     # percent units

print(f"Starter file: {len(df):,} pages")
print(f"Eligible pages (>=100 impressions in the earlier 30 days, position 1-20): {len(el):,}")

Starter file: 30,000 pages
Eligible pages (>=100 impressions in the earlier 30 days, position 1-20): 12,770


### Signal 1 (flag-linked): does CTR fall as position gets worse?

The CTR-fix flag assumes there is a sensible CTR for a given position, and that a fixed 0.5% cut-off separates "fine" from "under-capturing". Test: pooled CTR (total clicks / total impressions, so tiny pages don't dominate) per position band, with `n` printed, and what share of pages in each band sits under 0.5%.

In [ ]:
s1 = el.groupby("band", observed=True).agg(
    n=("content_id", "size"),
    clicks=("clicks_prev_30d", "sum"),
    impressions=("impressions_prev_30d", "sum"),
)
s1["pooled_ctr_pct"] = (100 * s1["clicks"] / s1["impressions"]).round(3)
s1["share_of_pages_under_0.5pct"] = (
    el.assign(under=el["ctr_prev"] < 0.5).groupby("band", observed=True)["under"].mean().round(3)
)
print("Total n in this table:", int(s1["n"].sum()))
s1[["n", "pooled_ctr_pct", "share_of_pages_under_0.5pct"]]

Total n in this table: 12770


,n,pooled_ctr_pct,share_of_pages_under_0.5pct
band,,,
1-3,480,0.557,0.735
4-6,3043,0.401,0.733
7-10,4379,0.257,0.809
11-15,2915,0.349,0.821
16-20,1953,0.351,0.833


**Verdict 1: MIXED.**

Position matters at the top: pooled CTR is about 0.56% at positions 1-3 and 0.40% at 4-6. But it does not keep falling: positions 7-10 are the lowest (about 0.26%), and 11-20 sit higher again (about 0.35%). I cannot explain that dip from this data, so I treat it as an observation, not a rule. The bigger finding for the flag: **73-83% of pages in every band are under 0.5%**, so the fixed cut-off mostly flags *typical* pages, not under-performers. Lesson for my rule: compare each page with its own position band, not with one global number.

### Signal 2: does a page that is far below its band's normal CTR stay that way?

My rule assumes the gap persists, otherwise a review today is wasted. For each eligible page: `ratio` = its earlier CTR divided by its band's typical earlier CTR. "Low later" = CTR in the **later** 30 days is under half of its band's typical CTR in that window (a label I defined; it is used only here, to check the signal, never as an input to the rule). Pages need at least 100 impressions in the later window to have a label.

In [ ]:
typ_prev = 100 * s1["clicks"] / s1["impressions"]                         # typical earlier CTR per band
el["typical_ctr_band"] = el["band"].map(typ_prev).astype(float)
el["ratio"] = el["ctr_prev"] / el["typical_ctr_band"]

# Hindsight outcome, only for pages with enough later volume. NOT an input to the rule.
ev = el[el["impressions_last_30d"] >= 100].copy()
ev["ctr_last"] = 100 * ev["clicks_last_30d"] / ev["impressions_last_30d"]
lt = ev.groupby("band", observed=True)[["clicks_last_30d", "impressions_last_30d"]].sum()
typ_last = 100 * lt["clicks_last_30d"] / lt["impressions_last_30d"]
ev["low_later"] = (ev["ctr_last"] < 0.5 * ev["band"].map(typ_last).astype(float)).astype(int)

ev["ratio_bucket"] = pd.cut(ev["ratio"], [-0.001, 0.25, 0.5, 1.0, np.inf],
                            labels=["<0.25", "0.25-0.5", "0.5-1.0", ">=1.0"])
s2 = ev.groupby("ratio_bucket", observed=True).agg(n=("low_later", "size"),
                                                  share_low_later=("low_later", "mean")).round(3)
print(f"Pages with a later-window label: {len(ev):,} of {len(el):,} eligible "
      f"({len(el) - len(ev):,} lost most of their impressions and have no label)")
print(f"Base rate (share low later, all labelled pages): {ev['low_later'].mean():.3f}")
s2

Pages with a later-window label: 11,111 of 12,770 eligible (1,659 lost most of their impressions and have no label)
Base rate (share low later, all labelled pages): 0.410


,n,share_low_later
ratio_bucket,,
<0.25,3513,0.672
0.25-0.5,1452,0.520
0.5-1.0,2430,0.323
>=1.0,3716,0.175


**Verdict 2: CONFIRMED.**

The share still low later falls in steps as the earlier ratio rises: about 67% for pages under a quarter of their band's norm, 52% for 0.25-0.5, 32% for 0.5-1.0 and 17% for pages at or above the norm, against a 41% base rate. This is directional evidence that the gap persists. It does **not** show that a rewrite would fix it: a page can stay low because of what people search or how the results page looks, which I cannot see here.

### The rule, in plain words

> A page is worth a title-and-meta review if it showed up enough in the earlier 30 days that its position band's typical CTR would have given it **at least 10 clicks**, and it got **less than half** of what that band normally gets. Rank by **missing clicks**: expected clicks at the band's typical CTR minus the clicks it actually got.

- **Score:** missing clicks (clicks it would have had at the band norm, minus clicks it had). No fitted weights.
- **Reason code (one):** `ctr_well_below_band_typical`
- **Action label:** `review_title_and_meta` (look at the search snippet and intent match; rewrite, or mark "monitor")
- **Inputs:** `impressions_prev_30d`, `clicks_prev_30d`, `avg_position` only. The "at least 10 expected clicks" floor is my hand-chosen guard so a page is not flagged on a few clicks; I test other floors below.

## 2. Build the ranked queue (writes the CSV)

The code below scores every eligible page, keeps the flagged ones, ranks them by missing clicks (ties broken by `content_id` so the order is the same on every run), writes `work/outputs/baseline_action_score.csv`, then checks it with precision@K. That CSV stays out of git by design; this notebook rebuilds it.

In [ ]:
FLOOR = 10   # minimum expected clicks at the band's typical CTR

def build_queue(frame, floor):
    q = frame.copy()
    q["expected_clicks"] = q["impressions_prev_30d"] * q["typical_ctr_band"] / 100
    q["missing_clicks"] = q["expected_clicks"] - q["clicks_prev_30d"]
    q["flag"] = (q["ratio"] < 0.5) & (q["expected_clicks"] >= floor)
    q["score"] = np.where(q["flag"], q["missing_clicks"], 0.0)
    out = q[q["flag"]].sort_values(["score", "content_id"], ascending=[False, True]).reset_index(drop=True)
    out["rank"] = out.index + 1
    out["reason_code"] = "ctr_well_below_band_typical"
    out["action"] = "review_title_and_meta"
    return out

queue = build_queue(el, FLOOR)

CSV_COLS = ["rank", "content_id", "client_id", "score", "reason_code", "action", "avg_position", "band",
            "impressions_prev_30d", "clicks_prev_30d", "ctr_prev", "typical_ctr_band",
            "expected_clicks", "missing_clicks"]
CSV_PATH = "work/outputs/baseline_action_score.csv"
queue[CSV_COLS].to_csv(CSV_PATH, index=False)

print(f"Eligible pages: {len(el):,}   Flagged (in the queue): {len(queue):,} ({len(queue)/len(el):.1%})")
print(f"Wrote {CSV_PATH}  ({len(queue):,} rows)")
queue[CSV_COLS].head(5).round(2)

Eligible pages: 12,770   Flagged (in the queue): 1,012 (7.9%)
Wrote work/outputs/baseline_action_score.csv  (1,012 rows)


,rank,content_id,client_id,score,reason_code,action,avg_position,band,impressions_prev_30d,clicks_prev_30d,ctr_prev,typical_ctr_band,expected_clicks,missing_clicks
0,1,content_5fe46e04994d,client_4e07408562,626.79,ctr_well_below_band_typical,review_title_and_meta,4.2,4-6,218786,250,0.11,0.40,876.79,626.79
1,2,content_8451fc6f034d,client_d029fa3a95,513.38,ctr_well_below_band_typical,review_title_and_meta,2.3,1-3,97606,30,0.03,0.56,543.38,513.38
2,3,content_8c19996aa890,client_4e07408562,508.87,ctr_well_below_band_typical,review_title_and_meta,2.5,1-3,161284,389,0.24,0.56,897.87,508.87
3,4,content_cb112fce36be,client_19581e27de,327.94,ctr_well_below_band_typical,review_title_and_meta,5.6,4-6,124500,171,0.14,0.40,498.94,327.94
4,5,content_73c54f78c06a,client_f369cb89fc,305.53,ctr_well_below_band_typical,review_title_and_meta,4.7,4-6,97200,84,0.09,0.40,389.53,305.53


In [ ]:
# precision@K: of the top K the rule flags, how many were still low in the later 30 days?
hind = ev.set_index("content_id")["low_later"]

def p_at_k(ids, k):
    labelled = pd.Series(list(ids)[:k]).map(hind).dropna()
    return round(float(labelled.mean()), 3), len(labelled)

base_rate = float(ev["low_later"].mean())
big_first = el.sort_values(["impressions_prev_30d", "content_id"], ascending=[False, True])["content_id"]

rows = []
for k in (10, 20, 50, 100):
    rule_p, rule_n = p_at_k(queue["content_id"], k)
    big_p, _ = p_at_k(big_first, k)
    rows.append({"K": k, "my rule": rule_p, "biggest pages first": big_p,
                 "base rate": round(base_rate, 3), "n labelled in top K": rule_n})
prec = pd.DataFrame(rows).set_index("K")

fixed = ev[ev["ctr_prev"] < 0.5]
print(f"Fixed rule 'earlier CTR < 0.5%': flags {len(fixed)/len(ev):.0%} of labelled pages, "
      f"{fixed['low_later'].mean():.3f} of them low later (not a ranking, so no K)")
prec

Fixed rule 'earlier CTR < 0.5%': flags 79% of labelled pages, 0.483 of them low later (not a ranking, so no K)


,my rule,biggest pages first,base rate,n labelled in top K
K,,,,
10,0.90,0.40,0.41,10
20,0.95,0.45,0.41,20
50,0.82,0.44,0.41,50
100,0.81,0.29,0.41,100


In [ ]:
# How much does the hand-picked floor matter?
sens = []
for fl in (5, 10, 25):
    qf = build_queue(el, fl)
    p50, _ = p_at_k(qf["content_id"], 50)
    p100, _ = p_at_k(qf["content_id"], 100)
    sens.append({"min expected clicks": fl, "pages flagged": len(qf), "precision@50": p50, "precision@100": p100})
print(pd.DataFrame(sens).to_string(index=False))

# Receipts (no IDs, safe to commit)
metrics = {
    "eligible_pages": int(len(el)), "flagged_pages": int(len(queue)), "floor_expected_clicks": FLOOR,
    "labelled_pages": int(len(ev)), "base_rate_low_later": round(base_rate, 4),
    "precision_at_k": {str(k): float(prec.loc[k, "my rule"]) for k in prec.index},
    "biggest_pages_first_at_k": {str(k): float(prec.loc[k, "biggest pages first"]) for k in prec.index},
    "fixed_rule_flag_share": round(len(fixed) / len(ev), 4),
    "fixed_rule_precision": round(float(fixed["low_later"].mean()), 4),
    "signal_verdicts": {"ctr_vs_position_flag_linked": "MIXED", "gap_persists": "CONFIRMED"},
}
with open("work/outputs/w04_baseline_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)
print("\nWrote work/outputs/w04_baseline_metrics.json")

 min expected clicks  pages flagged  precision@50  precision@100
                   5           1731          0.82           0.81
                  10           1012          0.82           0.81
                  25            435          0.82           0.81

Wrote work/outputs/w04_baseline_metrics.json


**Reading it:** the rule's top 50 are still low in the later window far more often than the 41% base rate, and far more often than just taking the biggest pages first. It also beats the fixed "CTR < 0.5%" rule, which is right only about 48% of the time across 79% of pages. The floor changes how long the queue is (1,731, 1,012 or 435 pages) but not the top of it, because ranking by missing clicks already puts big gaps first. This is the number my week-5 model has to beat: precision@50 for this frozen rule. I will not change the rule once the model work starts.

## 3. Top-10 review

For each of the top ten: the action, why it is there, and what would make it wrong. The last two columns of the table below (`ctr_last`, `low_later`) are **hindsight** from the later 30 days; they were not used to rank anything.

In [ ]:
top10 = queue.head(10).merge(ev[["content_id", "ctr_last", "low_later"]], on="content_id", how="left")
top10[["rank", "content_id", "client_id", "avg_position", "impressions_prev_30d", "clicks_prev_30d",
       "ctr_prev", "typical_ctr_band", "missing_clicks", "main_intent", "days_since_last_update",
       "content_age_days", "ctr_last", "low_later"]].round(2)

,rank,content_id,client_id,avg_position,impressions_prev_30d,clicks_prev_30d,ctr_prev,typical_ctr_band,missing_clicks,main_intent,days_since_last_update,content_age_days,ctr_last,low_later
0,1,content_5fe46e04994d,client_4e07408562,4.2,218786,250,0.11,0.40,626.79,informational,104,537,0.18,1
1,2,content_8451fc6f034d,client_d029fa3a95,2.3,97606,30,0.03,0.56,513.38,informational,20,280,0.02,1
2,3,content_8c19996aa890,client_4e07408562,2.5,161284,389,0.24,0.56,508.87,informational,20,445,0.28,1
3,4,content_cb112fce36be,client_19581e27de,5.6,124500,171,0.14,0.40,327.94,transactional,104,126,0.18,1
4,5,content_73c54f78c06a,client_f369cb89fc,4.7,97200,84,0.09,0.40,305.53,informational,20,97,0.10,1
5,6,content_c8e9d6ab9013,client_19581e27de,9.7,111885,0,0.00,0.26,287.77,informational,104,362,0.00,1
6,7,content_a7427266c305,client_19581e27de,5.7,78813,73,0.09,0.40,242.85,informational,104,257,0.10,1
7,8,content_2db251d1a841,client_f369cb89fc,5.6,84550,114,0.13,0.40,224.84,informational,20,97,0.21,0
8,9,content_36ff89c8214e,client_19581e27de,7.3,106412,63,0.06,0.26,210.69,informational,104,144,0.03,1
9,10,content_c84a0ab98e90,client_f369cb89fc,7.8,84773,19,0.02,0.26,199.04,informational,20,95,0.04,1


| Rank | Page | Action | Why it is here | What would make it wrong |
|---|---|---|---|---|
| 1 | `content_5fe4...` | review_title_and_meta | 218,786 impressions at position 4.2 got 250 clicks (0.11%); the 4-6 band normally gets 0.40%, so about 627 clicks are missing in 30 days. | Most of the impressions could come from one head query where people get their answer on the results page and do not click. A new title would not change that, and this data has no query detail to check. |
| 2 | `content_8451...` | review_title_and_meta | 97,606 impressions at position 2.3 got 30 clicks (0.03%) against 0.56% for positions 1-3: about 513 missing. | A CTR this low at a top-3 position looks more like a measurement or results-page problem than a bad title, and an average position can hide where the impressions really come from. The page was also updated 20 days ago, after the earlier window, so it has already changed. |
| 3 | `content_8c19...` | review_title_and_meta | 161,284 impressions at position 2.5 got 389 clicks (0.24%) against 0.56%: about 509 missing. | The top-3 benchmark (0.56%) rests on only 480 pages and may be too high for this page's mix of searches. Also updated 20 days ago. |
| 4 | `content_cb11...` | review_title_and_meta | 124,500 impressions at position 5.6 got 171 clicks (0.14%) against 0.40%: about 328 missing. | Intent is transactional. Shopping searches tend to show ads and boxes above the results, so a lower organic CTR can be normal. It is also one of four of my top ten from the same client. |
| 5 | `content_73c5...` | review_title_and_meta | 97,200 impressions at position 4.7 got 84 clicks (0.09%) against 0.40%: about 306 missing. | The page is only 97 days old and was updated 20 days ago, so the earlier 30 days may not describe the page as it is now. |
| 6 | `content_c8e9...` | review_title_and_meta | 111,885 impressions at position 9.7 got 0 clicks (and 0 sessions) against 0.26%: about 288 missing. | Zero clicks on 111k impressions looks like a tracking or reporting gap, not a weak title. Check the data source before touching the page. |
| 7 | `content_a742...` | review_title_and_meta | 78,813 impressions at position 5.7 got 73 clicks (0.09%) against 0.40%: about 243 missing. | The cause could be site-wide for this client (four of my top ten come from it), so the right fix may be one template or snippet change, not page-by-page rewrites. |
| 8 | `content_2db2...` | review_title_and_meta | 84,550 impressions at position 5.6 got 114 clicks (0.13%) against 0.40%: about 225 missing. | Updated 20 days ago, after the earlier window, so a fix may already be in. In hindsight the later CTR is 0.21%, just above my cut-off, so this is my clearest weak pick. |
| 9 | `content_36ff...` | review_title_and_meta | 106,412 impressions at position 7.3 got 63 clicks (0.06%) against 0.26%: about 211 missing. | The 7-10 band's typical CTR (0.26%) is the lowest band in this slice, which signal 1 could not explain. If that benchmark is too low or too high for this band, the gap is not the size shown. |
| 10 | `content_c84a...` | review_title_and_meta | 84,773 impressions at position 7.8 got 19 clicks (0.02%) against 0.26%: about 199 missing. | Same doubt about the 7-10 benchmark as rank 9, and the page was updated 20 days ago. |

## 4. Weak picks + leakage check

**Weak picks.** Rank 8 is the clearest: its CTR in the later window moved back above my "low" cut-off, and it had been updated shortly before. Ranks 2 and 6 are weak in a different way: a top-3 page with a 0.03% CTR, and a page with 111k impressions and zero clicks, look more like data problems than title problems, so the action may be "check the data" before "rewrite". Seven of the top ten come from two clients, so a few clients drive the queue and the real fix may sit at client level.

**Leakage check.** The cells below check that no later-window, label-derived or product-flag column feeds the rule, and that the written CSV carries none.

In [ ]:
RULE_INPUTS = ["impressions_prev_30d", "clicks_prev_30d", "avg_position"]
FORBIDDEN = ["impressions_last_30d", "clicks_last_30d", "sessions_last_30d", "trend_direction", "trend_pct",
             "is_declining_label", "ctr", "clicks_90d", "impressions_90d", "ctr_last", "low_later"]
leaked = [c for c in RULE_INPUTS if c in FORBIDDEN or "last_30d" in c]
assert not leaked, f"future-window or label-derived input found: {leaked}"

written = pd.read_csv(CSV_PATH)
assert not (set(written.columns) & set(FORBIDDEN)), "the CSV carries a later-window or label column"
print("Rule inputs:", RULE_INPUTS)
print("Leakage guard passed: no later-window, trend or label column feeds the rule or sits in the CSV.")

# Sensitivity 1: avg_position is a 90-day average and overlaps the later 30 days.
# Does the rule still work with NO position at all (one pooled benchmark for everyone)?
t0 = 100 * el["clicks_prev_30d"].sum() / el["impressions_prev_30d"].sum()
nopos = el.copy()
nopos["typical_ctr_band"] = t0
nopos["ratio"] = nopos["ctr_prev"] / t0
nopos_q = build_queue(nopos, FLOOR)
print(f"\nWith position bands:  precision@50 = {p_at_k(queue['content_id'], 50)[0]},  @100 = {p_at_k(queue['content_id'], 100)[0]}")
print(f"Position-free rule:   precision@50 = {p_at_k(nopos_q['content_id'], 50)[0]},  @100 = {p_at_k(nopos_q['content_id'], 100)[0]}")

# Sensitivity 2: concentration
vc = top10["client_id"].value_counts()
print(f"\nTop 10: {top10['client_id'].nunique()} distinct clients; the two most repeated hold {int(vc.iloc[:2].sum())} of 10")
print(f"Top 100: largest client's share = {queue.head(100)['client_id'].value_counts(normalize=True).iloc[0]:.0%}")

Rule inputs: ['impressions_prev_30d', 'clicks_prev_30d', 'avg_position']
Leakage guard passed: no later-window, trend or label column feeds the rule or sits in the CSV.

With position bands:  precision@50 = 0.82,  @100 = 0.81
Position-free rule:   precision@50 = 0.84,  @100 = 0.84

Top 10: 4 distinct clients; the two most repeated hold 7 of 10
Top 100: largest client's share = 63%


**What the checks say.** No later-window or label-derived column feeds the rule or sits in the CSV. `avg_position` is the one input I cannot fully clean: it is a 90-day average that overlaps the later 30 days. The position-free variant scores about the same, so position overlap does not look like it is driving the result, though position bands still help the benchmark be fair (signal 1). The 41% base rate and the 82% precision@50 come only from pages that kept at least 100 impressions in the later window; pages that lost most of their traffic have no label.

**What I can and cannot claim.** *Observed:* pages the rule ranks high are still below their band's normal CTR a month later much more often than other pages. *Decision-support:* it gives a reviewer a short, readable queue with reasons. *Not claimed:* that a rewrite raises CTR (this data cannot show it), or that the queue is right for any client, since one client holds a large share of the pages.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.